### Data Engineering Lifecycle

#### Objetivo Técnico

Criar um processo em ETL de dados médicos do Synthea em arquitetura medalhão, demonstrando como o pysparke e o SQL podem ser incorporados para evoluir a qualidade desta tarefa.

#### Objetivo Conceitual ou de Negócios

Qual o custo médio de tratamento por condição médica, considerando a demografia dos pacientes e a eficiência das organizações de saúde?

#### Configuração e Leitura (Bronze)

In [0]:
# Verificando as tabelas no catálogo
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA synthea_data")
display(spark.catalog.listTables())

name,catalog,namespace,description,tableType,isTemporary
conditions,workspace,List(synthea_data),Created by the file upload UI,MANAGED,false
encounters,workspace,List(synthea_data),Created by the file upload UI,MANAGED,false
gold_disease_analytics,workspace,List(synthea_data),null,MANAGED,false
patients,workspace,List(synthea_data),Created by the file upload UI,MANAGED,false
silver_encounters,workspace,List(synthea_data),null,MANAGED,false
silver_patients,workspace,List(synthea_data),null,MANAGED,false


In [0]:
%sql
-- Descobrindo onde o Databricks escondeu os arquivos
DESCRIBE EXTENDED silver_patients;

col_name,data_type,comment
patient_id,string,null
BIRTHDATE,date,null
CITY,string,null
STATE,string,null
RACE,string,null
GENDER,string,null
age,int,null
age_group,string,null
,,
# Delta Statistics Columns,,


In [0]:
spark.sql("DROP TABLE IF EXISTS silver_patients")
spark.sql("DROP TABLE IF EXISTS silver_encounters")
spark.sql("DROP TABLE IF EXISTS gold_disease_analytics")

DataFrame[]

In [0]:

# 1. Leitura das Tabelas do Catálogo (Camada Bronze/Raw)
df_conditions_raw = spark.table("conditions")
df_encounters_raw = spark.table("encounters")
df_patients_raw = spark.table("patients")

# 3. Análise Volumétrica (Agora das 3 tabelas)
print("--- Volumetria Inicial ---")
print(f"Pacientes: {df_patients_raw.count()}")
print(f"Encontros Médicos: {df_encounters_raw.count()}")
print(f"Condições/Doenças: {df_conditions_raw.count()}")

# 4. Inspeção Visual Rápida
display(df_patients_raw.limit(5))
display(df_encounters_raw.limit(5))
display(df_conditions_raw.limit(5))

--- Volumetria Inicial ---
Pacientes: 111
Encontros Médicos: 5924
Condições/Doenças: 4140


Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,1987-01-21,null,999-35-6397,S99957832,X22616959X,Mr.,Britt177,Clifton91,Keeling57,null,null,M,white,nonhispanic,M,Waltham Massachusetts US,840 Grimes Well Apt 27,Duxbury,Massachusetts,Plymouth County,25023,2332,42.068929818471844,-70.70631994468269,94868.89,24033.16,60192
edc17058-55fb-08c7-12df-ece93a402e50,1986-03-31,null,999-75-7017,S99916291,X68816273X,Mr.,Tobias236,Yong583,Harris789,null,null,M,white,nonhispanic,M,New Bedford Massachusetts US,166 Funk Burg,Gardner,Massachusetts,Worcester County,25027,1440,42.568845372210774,-71.96183525180287,161214.72,24287.84,44043
80e7f50a-3e99-d5ac-cf97-f8a4b4f9e6c7,2006-02-17,null,999-63-2435,S99916556,null,Ms.,Lu473,null,Schimmel440,null,null,null,white,nonhispanic,F,Naples Campania IT,218 Hodkiewicz Route,Ludlow,Massachusetts,Hampden County,null,0,42.148873574699984,-72.49746261627895,5063.58,89666.75,16637
782001bc-f712-50ae-04f5-9a488f3ef4aa,1991-10-20,null,999-72-6313,S99963096,X63092846X,Ms.,Hortencia577,Francoise850,Renner328,null,null,S,white,hispanic,F,Weymouth Massachusetts US,113 Dooley Extension Unit 99,Hampden,Massachusetts,Hampden County,null,0,42.110805436535486,-72.40585815841523,90525.56,1391754.02,33549
30e48e16-2df7-207e-7a3d-1650ef0c1ed8,1956-06-10,1961-04-13,999-50-1537,null,null,null,Caryl47,Elenore794,Cassin499,null,null,null,white,nonhispanic,F,Lancaster Massachusetts US,1084 Zulauf Park,Bourne,Massachusetts,Barnstable County,25001,2532,41.765941720531565,-70.61634440837028,14631.18,848.66,145499


Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
8151e1b2-5789-578d-3e3a-d1d9b7c142ee,2005-03-16T06:04:53.000Z,2005-03-16T06:54:52.000Z,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2936ca90-6594-357d-be14-53fe0ab84dd1,0d67d251-73f5-3118-be75-41f33e95b7d1,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,162673000,General examination of patient (procedure),136.8,840.2,0.0,null,null
6dce0dad-f85f-8a26-9583-f1bcdde4efc1,2006-03-22T06:04:53.000Z,2006-03-22T06:35:36.000Z,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2936ca90-6594-357d-be14-53fe0ab84dd1,0d67d251-73f5-3118-be75-41f33e95b7d1,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,wellness,162673000,General examination of patient (procedure),136.8,1385.72,0.0,null,null
6f2cf873-f64e-74a7-efba-6d772f418395,2012-03-28T06:04:53.000Z,2012-03-28T06:54:32.000Z,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2936ca90-6594-357d-be14-53fe0ab84dd1,0d67d251-73f5-3118-be75-41f33e95b7d1,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,wellness,162673000,General examination of patient (procedure),136.8,919.9,0.0,null,null
87741be8-2223-14c5-375d-976414fcd9aa,2016-04-11T11:14:53.000Z,2016-04-11T12:14:53.000Z,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,52efa2e5-007f-3396-a83a-28ca015f0595,3169b71a-aa09-3f9e-8ca4-a592bb52e8aa,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,emergency,50849002,Emergency room admission (procedure),146.18,16389.19,8406.1,312608009,Laceration - injury (disorder)
1f9f55cd-ae88-b9ab-15e1-927e73848ca3,2017-12-22T02:04:53.000Z,2017-12-22T02:19:53.000Z,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,3ff0f58b-7a1f-3334-9449-ff15f68b7d86,5f024f16-dc88-3249-8eef-e04d27a7d717,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,ambulatory,185345009,Encounter for symptom (procedure),85.55,85.55,0.0,195662009,Acute viral pharyngitis (disorder)


START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
2005-03-16,null,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,8151e1b2-5789-578d-3e3a-d1d9b7c142ee,http://snomed.info/sct,224299000,Received higher education (finding)
2006-03-22,null,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,6dce0dad-f85f-8a26-9583-f1bcdde4efc1,http://snomed.info/sct,266948004,Has a criminal record (finding)
2006-03-22,2021-04-07,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,6dce0dad-f85f-8a26-9583-f1bcdde4efc1,http://snomed.info/sct,361055000,Misuses drugs (finding)
2012-03-28,2018-04-04,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,6f2cf873-f64e-74a7-efba-6d772f418395,http://snomed.info/sct,160904001,Part-time employment (finding)
2012-03-28,null,8c8e1c9a-b310-43c6-33a7-ad11bad21c40,6f2cf873-f64e-74a7-efba-6d772f418395,http://snomed.info/sct,706893006,Victim of intimate partner abuse (finding)


#### Tratamento Silver (Patients)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Definição da lógica de transformação
df_patients_silver = df_patients_raw.select(
    F.col("Id").alias("patient_id"),
    F.col("BIRTHDATE"),
    F.col("CITY"),
    F.col("STATE"),
    F.col("RACE"),
    F.col("GENDER"),
    # Cálculo de idade precisa
    F.floor(F.datediff(F.current_date(), F.col("BIRTHDATE")) / 365.25).cast(IntegerType()).alias("age")
).withColumn(
    "age_group",
    F.when(F.col("age") < 18, "0-17 Pediatric")
     .when((F.col("age") >= 18) & (F.col("age") < 60), "18-59 Adult")
     .otherwise("60+ Senior")
)

In [0]:
# Persistindo a Silver como Tabela Delta (Melhor prática: salvar resultados intermediários estáveis)
df_patients_silver.write.format("delta").mode("overwrite").saveAsTable("silver_patients")

In [0]:
display(spark.read.table("silver_patients").groupBy("age_group").count())

age_group,count
18-59 Adult,68
60+ Senior,24
0-17 Pediatric,19


#### Tratamento Silver (Encounters)

Objetivo: Analisar a jornada do paciente. Cenário: Queremos saber quantos dias se passaram desde a última consulta desse paciente. Isso exige olhar para trás nos dados (Window Function lag).

In [0]:
from pyspark.sql import functions as F

# 1. Limpeza, Tipagem e Colunas Derivadas Simples (Sem olhar para trás)
df_encounters_clean = df_encounters_raw.select(
    F.col("Id").alias("encounter_id"),
    F.col("PATIENT").alias("patient_id"),
    F.col("START").cast("timestamp"),
    F.col("STOP").cast("timestamp"),
    F.col("ENCOUNTERCLASS"),
    F.col("CODE").alias("encounter_code"),
    F.col("TOTAL_CLAIM_COST").cast("double"),
    F.col("PAYER_COVERAGE").cast("double"),
    F.col("REASONDESCRIPTION")
).withColumn(
    "duration_minutes", 
    (F.unix_timestamp("STOP") - F.unix_timestamp("START")) / 60
)

# 2. Criando a Silver (Direto, sem janelamento)
df_encounters_silver = df_encounters_clean

# Salvando Silver Encounters
df_encounters_silver.write.format("delta").mode("overwrite").saveAsTable("silver_encounters")

# Display simplificado (apenas colunas que existem)
display(df_encounters_silver.select("patient_id", "START", "duration_minutes", "TOTAL_CLAIM_COST").limit(10))

patient_id,START,duration_minutes,TOTAL_CLAIM_COST
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2005-03-16T06:04:53.000Z,49.983333333333334,840.2
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2006-03-22T06:04:53.000Z,30.716666666666665,1385.72
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2012-03-28T06:04:53.000Z,49.65,919.9
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2016-04-11T11:14:53.000Z,60.0,16389.19
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2017-12-22T02:04:53.000Z,15.0,85.55
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2018-02-24T11:48:33.000Z,60.0,14153.09
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2018-04-04T06:04:53.000Z,47.416666666666664,914.78
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2021-04-07T06:04:53.000Z,41.68333333333333,989.36
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2022-04-07T04:04:53.000Z,15.0,85.55
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2024-04-10T06:04:53.000Z,42.266666666666666,1166.87


In [0]:

# Salvando Silver Encounters
df_encounters_silver.write.format("delta").mode("overwrite").saveAsTable("silver_encounters")

In [0]:
display(df_encounters_silver.select(
    "patient_id", 
    "START", 
    "duration_minutes", 
    "TOTAL_CLAIM_COST"
).limit(10))

patient_id,START,duration_minutes,TOTAL_CLAIM_COST
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2005-03-16T06:04:53.000Z,49.983333333333334,840.2
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2006-03-22T06:04:53.000Z,30.716666666666665,1385.72
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2012-03-28T06:04:53.000Z,49.65,919.9
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2016-04-11T11:14:53.000Z,60.0,16389.19
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2017-12-22T02:04:53.000Z,15.0,85.55
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2018-02-24T11:48:33.000Z,60.0,14153.09
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2018-04-04T06:04:53.000Z,47.416666666666664,914.78
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2021-04-07T06:04:53.000Z,41.68333333333333,989.36
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2022-04-07T04:04:53.000Z,15.0,85.55
8c8e1c9a-b310-43c6-33a7-ad11bad21c40,2024-04-10T06:04:53.000Z,42.266666666666666,1166.87


#### Comparativo SQL vs PySpark (Interatividade)

Objetivo: Provar que SQL e PySpark geram o mesmo plano de execução no Databricks.

In [0]:
# Célula PySpark: Análise de Custo Médio por Classe
df_agg_py = spark.read.table("silver_encounters") \
    .groupBy("ENCOUNTERCLASS") \
    .agg(F.avg("TOTAL_CLAIM_COST").alias("avg_cost")) \
    .orderBy(F.desc("avg_cost"))

display(df_agg_py)

ENCOUNTERCLASS,avg_cost
inpatient,21095.970117647066
hospice,14805.129999999997
snf,14187.187857142857
ambulatory,3639.1649597197666
emergency,3089.9860869565223
outpatient,1308.8858898305098
wellness,1063.5513847305394
urgentcare,953.5085393258419
virtual,315.7


In [0]:
df_agg_py.explain() 

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonSort [avg_cost#21009 DESC NULLS LAST]
         +- PhotonGroupingAgg(keys=[ENCOUNTERCLASS#21100], functions=[avg(TOTAL_CLAIM_COST#21102)])
            +- PhotonScan parquet workspace.synthea_data.silver_encounters[ENCOUNTERCLASS#21100,TOTAL_CLAIM_COST#21102] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-1zmfu/uc/b233e871-6472-482c-af3a-cf4200701874..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<ENCOUNTERCLASS:string,TOTAL_CLAIM_COST:double>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = silver_encounters



In [0]:
%sql
-- Célula SQL: O mesmo resultado (Magics %sql)

SELECT 
    ENCOUNTERCLASS,
    AVG(TOTAL_CLAIM_COST) as avg_cost
FROM silver_encounters
GROUP BY ENCOUNTERCLASS
ORDER BY avg_cost DESC

ENCOUNTERCLASS,avg_cost
inpatient,21095.970117647066
hospice,14805.129999999997
snf,14187.187857142857
ambulatory,3639.1649597197666
emergency,3089.9860869565223
outpatient,1308.8858898305098
wellness,1063.5513847305394
urgentcare,953.5085393258419
virtual,315.7


##### Otimização de Joins

Objetivo: Cruzar tabelas grandes (Encounters) com tabelas pequenas (Patients) usando Broadcast. Cenário: Analisar custos por Raça e Cidade.

#### Camada Gold - Agregação para Business Intelligence

Objetivo: Criar uma tabela pronta para ser consumida por PowerBI/Tableau ou Databricks SQL Dashboards. Regra de Negócio: Quais cidades têm o maior custo médio de atendimento para Idosos (60+)?

#### Criação de Visualização

Clique no sinal de + acima da tabela de resultados -> Visualization.

Configuração do Gráfico:

Type: Bar Chart.

X Column: CITY.

Y Column: total_spent.

Group/Color By: RACE.

Title: Custo de Saúde Sênior por Cidade e Etnia.

In [0]:
# Lendo as tabelas silver
df_enc = spark.read.table("silver_encounters")
df_pat = spark.read.table("silver_patients")

# JOIN OTIMIZADO (Broadcast)
# df_joined = df_enc.join(
#     F.broadcast(df_pat), 
#     df_enc.patient_id == df_pat.patient_id, 
#     "inner"
# )

df_joined = df_enc.join(
    df_pat, 
    df_enc.patient_id == df_pat.patient_id, 
    "inner"
)

# Preparando dados para a Gold
df_enriched = df_joined.select(
    df_enc["*"],
    df_pat["RACE"],
    df_pat["CITY"],
    df_pat["age_group"]
)

# Agregação da Gold
df_gold = df_enriched.filter(F.col("age_group") == "60+ Senior") \
    .groupBy("CITY", "RACE") \
    .agg(
        F.count("encounter_id").alias("total_visits"),
        F.sum("TOTAL_CLAIM_COST").alias("total_spent"),
        F.avg("duration_minutes").alias("avg_duration_min")
    ) \
    .withColumn("avg_cost_per_visit", F.col("total_spent") / F.col("total_visits")) \
    .orderBy(F.desc("total_spent"))

# Salvando Gold particionada
df_gold.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("CITY") \
    .saveAsTable("gold_disease_analytics")

display(df_gold)

CITY,RACE,total_visits,total_spent,avg_duration_min,avg_cost_per_visit
Lawrence,white,624,1532610.2499999998,358.47339743589725,2456.1061698717945
Bourne,white,227,708329.2199999994,667.173348017621,3120.3930396475744
Winchester,black,471,703227.6099999989,76.37186836518052,1493.0522505307833
Templeton,white,273,566381.239999999,84.02814407814415,2074.656556776553
Georgetown,white,178,487602.0299999997,508.12911985018707,2739.3372471910093
Norton Center,white,504,428547.19000000047,209.23333333333335,850.2920436507945
Worcester,white,91,252540.06999999986,2219.2985347985345,2775.1656043956027
Falmouth,white,102,231846.34000000003,304.85669934640526,2273.0033333333336
Cambridge,white,49,173866.77000000005,227.48503401360543,3548.3014285714294
Saugus,black,44,153979.68000000002,604.7215909090911,3499.5381818181822


Databricks visualization. Run in Databricks to view.

In [0]:
df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonBroadcastHashJoin [patient_id#23973], [patient_id#23982], Inner, BuildRight, false, true
         :- PhotonScan parquet workspace.synthea_data.silver_encounters[encounter_id#23972,patient_id#23973,START#23974,STOP#23975,ENCOUNTERCLASS#23976,encounter_code#23977L,TOTAL_CLAIM_COST#23978,PAYER_COVERAGE#23979,REASONDESCRIPTION#23980,duration_minutes#23981] DataFilters: [isnotnull(patient_id#23973)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-1zmfu/uc/b233e871-6472-482c-af3a-cf4200701874..., OptionalDataFilters: [hashedrelationcontains(patient_id#23973)], PartitionFilters: [], ReadSchema: struct<encounter_id:string,patient_id:string,START:timestamp,STOP:timestamp,ENCOUNTERCLASS:string..., RequiredDataFilters: [isnotnull(patient_id#23973)]
         +- PhotonShuffleExchangeSource
            +- PhotonSh

#### Otimização com e sem Broadcast

In [0]:
import time

print("--- 1. FORÇANDO JOIN LENTO (SortMerge) ---")

# USAMOS .hint("merge") PARA PROIBIR O BROADCAST
# Isso força o Spark a fazer o Shuffle (SortMergeJoin)
df_joined_slow = df_enc.join(
    df_pat.hint("merge"), 
    df_enc.patient_id == df_pat.patient_id, 
    "inner"
)

# Medindo o tempo
start_time = time.time()
df_joined_slow.write.format("noop").mode("overwrite").save() # Action
end_time = time.time()

print(f"Tempo (SortMerge): {end_time - start_time:.4f} segundos")

# Mostre aos alunos que aqui aparece "SortMergeJoin" no plano
df_joined_slow.explain()

--- 1. FORÇANDO JOIN LENTO (SortMerge) ---
Tempo (SortMerge): 1.3106 segundos
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   SortMergeJoin [patient_id#22502], [patient_id#22519], Inner
   :- ColumnarToRow
   :  +- PhotonResultStage
   :     +- PhotonSort [patient_id#22502 ASC NULLS FIRST]
   :        +- PhotonShuffleExchangeSource
   :           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#15337]
   :              +- PhotonShuffleExchangeSink hashpartitioning(patient_id#22502, 1024)
   :                 +- PhotonScan parquet workspace.synthea_data.silver_encounters[encounter_id#22501,patient_id#22502,START#22503,STOP#22504,ENCOUNTERCLASS#22505,encounter_code#22506L,TOTAL_CLAIM_COST#22507,PAYER_COVERAGE#22508,REASONDESCRIPTION#22509,duration_minutes#22510] DataFilters: [isnotnull(patient_id#22502)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-1zmfu/uc/b233e871-6472-482c-af3a-cf4200701874...

In [0]:
import time
from pyspark.sql import functions as F

print("--- INICIANDO JOIN COM BROADCAST ---")

# 1. Join Otimizado
df_joined_fast = df_enc.join(
    F.broadcast(df_pat), 
    df_enc.patient_id == df_pat.patient_id, 
    "inner"
)

# 2. Medindo o tempo
start_time = time.time()
df_joined_fast.write.format("noop").mode("overwrite").save()
end_time = time.time()

print(f"Tempo de Execução (Broadcast): {end_time - start_time:.4f} segundos")

# 3. Mostrando o Plano Físico (Procure por 'BroadcastHashJoin')
df_joined_fast.explain()

--- INICIANDO JOIN COM BROADCAST ---
Tempo de Execução (Broadcast): 0.5973 segundos
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonBroadcastHashJoin [patient_id#22661], [patient_id#22678], Inner, BuildRight, false, true
         :- PhotonScan parquet workspace.synthea_data.silver_encounters[encounter_id#22660,patient_id#22661,START#22662,STOP#22663,ENCOUNTERCLASS#22664,encounter_code#22665L,TOTAL_CLAIM_COST#22666,PAYER_COVERAGE#22667,REASONDESCRIPTION#22668,duration_minutes#22669] DataFilters: [isnotnull(patient_id#22661)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-1zmfu/uc/b233e871-6472-482c-af3a-cf4200701874..., OptionalDataFilters: [hashedrelationcontains(patient_id#22661)], PartitionFilters: [], ReadSchema: struct<encounter_id:string,patient_id:string,START:timestamp,STOP:timestamp,ENCOUNTERCLASS:string..., RequiredDataFilters: [isnotnul